# 1. ติดตั้งไลบรารีที่จำเป็นสำหรับ Google Colab
รันเซลล์นี้เพื่อติดตั้งไลบรารีที่ยังไม่มีใน Colab

In [ ]:
!pip install mne torchaudio scikit-learn numpy


In [ ]:
!gdown 1nRiS8hahGBsWmV8bqnIesDfZuJaVywly

Downloading...
From (original): https://drive.google.com/uc?id=1nRiS8hahGBsWmV8bqnIesDfZuJaVywly
From (redirected): https://drive.google.com/uc?id=1nRiS8hahGBsWmV8bqnIesDfZuJaVywly&confirm=t&uuid=1899444a-d7b5-4d3d-92db-cc3ed62f7ccb
To: /content/data-capstone.zip
100% 105M/105M [00:04<00:00, 22.7MB/s]


# 2. เตรียมข้อมูล (Upload หรือแตกไฟล์ .zip)
ถ้าคุณอัปโหลดไฟล์ `data.zip` มาไว้ที่ Colab แล้ว ให้รันคำสั่งด้านล่างเพื่อแตกไฟล์

In [ ]:
# สมมติว่าอัปโหลดไฟล์ data.zip มาแล้ว
!unzip -q "/content/data-capstone.zip" -d /content/data/
# ถึงตรงนี้คุณควรมีโฟลเดอร์ data/ ที่มีไฟล์ .edf เรียบร้อยแล้ว


replace /content/data/data-capstone/MDD S1 EC.edf? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


# 3. โค้ดหลัก (Imports & Config)

In [ ]:
import os
import glob
import numpy as np
import mne
import torch
import torch.nn as nn
import torchaudio
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import re

mne.set_log_level('WARNING')

DATA_DIR = '/content/data/data-capstone' # เปลี่ยน path ให้ตรงกับโฟลเดอร์ของคุณถ้าจำเป็น
EPOCH_DURATION = 5.0
FS_TARGET = 128
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 0.001
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")


Using device: cuda


# 4. ฟังก์ชันจัดการข้อมูล (Data Preparation & Splitting)

In [ ]:
def parse_subjects(data_dir):
    files = glob.glob(os.path.join(data_dir, '*.edf'))
    subjects = {}
    for f in files:
        basename = os.path.basename(f).replace('.edf', '')
        label = 1 if 'MDD' in basename else 0
        subject = re.sub(r'\s*E[CO].*', '', basename).strip()

        if subject not in subjects:
            subjects[subject] = {'label': label, 'files': []}
        subjects[subject]['files'].append(f)
    return subjects

def load_and_epoch_data(files_with_labels, epoch_duration, stride_duration, fs_target, is_train=False):
    X = []
    y = []
    target_channels = 19  # ล็อคไว้ที่ 19 ช่องสัญญาณมาตรฐานของ EEG (10-20 system)

    for f, label in files_with_labels:
        try:
            raw = mne.io.read_raw_edf(f, preload=True, verbose=False)

            # --- ส่วนที่แก้ไข: ลบช่องที่ไม่ใช่คลื่นสมองทิ้ง ---
            # ตัดช่อง stim หรือ annotations ทิ้งไป
            channels_to_drop = [ch for ch in raw.ch_names if 'stim' in ch.lower() or 'anno' in ch.lower() or 'event' in ch.lower()]
            if channels_to_drop:
                raw.drop_channels(channels_to_drop)
            # ----------------------------------------

            raw.resample(fs_target)
            data = raw.get_data() # (Channels, Time)

            # เอาแค่ 19 ช่องสัญญาณแรก (ตัดช่องขยะอื่นๆ ที่อาจจะต่อท้ายมาออก)
            if data.shape[0] >= target_channels:
                data = data[:target_channels, :]
            else:
                # ถ้าไฟล์ไหนช่องไม่ถึง 19 ให้ข้ามไปเลย จะได้ไม่มีการเติม 0 (Zero Padding) ที่จะทำให้โมเดลจำผิดๆ
                print(f"Skipping {f} because it only has {data.shape[0]} channels")
                continue

            mean = np.mean(data, axis=1, keepdims=True)
            std = np.std(data, axis=1, keepdims=True)
            data = (data - mean) / (std + 1e-8)

            samples_per_epoch = int(epoch_duration * fs_target)
            stride_samples = int(stride_duration * fs_target)

            for start in range(0, data.shape[1] - samples_per_epoch + 1, stride_samples):
                epoch = data[:, start:start+samples_per_epoch]

                if is_train:
                    noise = np.random.normal(0, 0.05, epoch.shape)
                    epoch_noisy = epoch + noise
                    X.append(epoch_noisy)
                    y.append(label)

                X.append(epoch)
                y.append(label)
        except Exception as e:
            print(f"Skipping {f} due to error: {e}")

    return np.array(X), np.array(y)


# 5. PyTorch Dataset & Model (STFT + CNN + LSTM)

In [ ]:
class EEGDataset(Dataset):
    def __init__(self, data, labels):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.stft = torchaudio.transforms.Spectrogram(
            n_fft=128,
            hop_length=32,
            power=2.0
        )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx]
        y = self.labels[idx]
        spec = self.stft(x)
        spec = torch.log1p(spec)
        return spec, y

class CNN_LSTM_Model(nn.Module):
    def __init__(self, num_channels, num_classes=2):
        super(CNN_LSTM_Model, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(num_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, None))
        self.lstm = nn.LSTM(input_size=32, hidden_size=64, num_layers=2, batch_first=True, dropout=0.3)
        self.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.cnn(x)
        x = self.adaptive_pool(x)
        x = x.squeeze(2)
        x = x.permute(0, 2, 1)
        lstm_out, _ = self.lstm(x)
        out = lstm_out[:, -1, :]
        out = self.fc(out)
        return out


# 6. Training Pipeline (รันการเทรน)

In [ ]:
print("1. Parsing Subjects...")
subjects = parse_subjects(DATA_DIR)
mdd_subs = [s for s, info in subjects.items() if info['label'] == 1]
hc_subs = [s for s, info in subjects.items() if info['label'] == 0]

print(f"Total Subjects -> MDD: {len(mdd_subs)}, Healthy: {len(hc_subs)}")

mdd_train, mdd_test = train_test_split(mdd_subs, test_size=0.2, random_state=42)
hc_train, hc_test = train_test_split(hc_subs, test_size=0.2, random_state=42)

train_file_labels = []
test_file_labels = []

for s in mdd_train + hc_train:
    for f in subjects[s]['files']:
        train_file_labels.append((f, subjects[s]['label']))

for s in mdd_test + hc_test:
    for f in subjects[s]['files']:
        test_file_labels.append((f, subjects[s]['label']))

print(f"2. Extracting Epochs and Augmenting...")
X_train, y_train = load_and_epoch_data(train_file_labels, EPOCH_DURATION, stride_duration=2.5, fs_target=FS_TARGET, is_train=True)
X_test, y_test = load_and_epoch_data(test_file_labels, EPOCH_DURATION, stride_duration=5.0, fs_target=FS_TARGET, is_train=False)

print(f"Train Shape: {X_train.shape}, Test Shape: {X_test.shape}")

train_dataset = EEGDataset(X_train, y_train)
test_dataset = EEGDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("3. Initializing Model...")
num_channels = X_train.shape[1]
model = CNN_LSTM_Model(num_channels=num_channels).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("4. Starting Training...")
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    correct = 0
    total = 0

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

    train_acc = 100 * correct / total

    model.eval()
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
            outputs = model(batch_x)
            _, predicted = torch.max(outputs.data, 1)
            test_total += batch_y.size(0)
            test_correct += (predicted == batch_y).sum().item()

    test_acc = 100 * test_correct / test_total
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {train_loss/len(train_loader):.4f} - Train Acc: {train_acc:.2f}% - Test Acc: {test_acc:.2f}%")


1. Parsing Subjects...
Total Subjects -> MDD: 30, Healthy: 15
2. Extracting Epochs and Augmenting...


/tmp/ipykernel_10806/3712484329.py:30: RuntimeWarning: Resampling of the stim channels caused event information to become unreliable. Consider finding events on the original data and passing the event matrix as a parameter.
  raw.resample(fs_target)


Train Shape: (15988, 19, 640), Test Shape: (947, 19, 640)
3. Initializing Model...
4. Starting Training...
Epoch [1/20] - Loss: 0.0428 - Train Acc: 98.69% - Test Acc: 87.43%
Epoch [2/20] - Loss: 0.0086 - Train Acc: 99.78% - Test Acc: 91.55%
Epoch [3/20] - Loss: 0.0139 - Train Acc: 99.61% - Test Acc: 91.24%
Epoch [4/20] - Loss: 0.0036 - Train Acc: 99.87% - Test Acc: 92.82%
Epoch [5/20] - Loss: 0.0047 - Train Acc: 99.83% - Test Acc: 89.33%
Epoch [6/20] - Loss: 0.0042 - Train Acc: 99.86% - Test Acc: 89.33%
Epoch [7/20] - Loss: 0.0015 - Train Acc: 99.96% - Test Acc: 89.02%
Epoch [8/20] - Loss: 0.0029 - Train Acc: 99.92% - Test Acc: 89.33%
Epoch [9/20] - Loss: 0.0009 - Train Acc: 99.97% - Test Acc: 89.44%
Epoch [10/20] - Loss: 0.0032 - Train Acc: 99.92% - Test Acc: 90.39%
Epoch [11/20] - Loss: 0.0018 - Train Acc: 99.96% - Test Acc: 90.18%
Epoch [12/20] - Loss: 0.0031 - Train Acc: 99.92% - Test Acc: 88.07%
Epoch [13/20] - Loss: 0.0036 - Train Acc: 99.88% - Test Acc: 90.07%
Epoch [14/20] - Lo

In [ ]:
torch.save(model.state_dict(), 'model_weights.pth')